# Sprint 5 — Validación final en test set

Este notebook usa el modelo final definido después de comparar desempeño técnico e impacto económico:

- **Modelo:** `LogisticRegression` tuneado.
- **Threshold operativo:** `0.70`.
- **Criterio final:** maximizar valor económico esperado manteniendo una detección relevante de `Bad Buys`.

El threshold `0.50` se mantiene como referencia técnica, pero el análisis económico mostró que `0.70` reduce de forma importante los falsos positivos y mejora el valor esperado del modelo.


In [8]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    precision_score,
    recall_score,
    fbeta_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

from src.config import *
from src.preprocessing import split_X_y

pd.set_option("display.max_columns", 120)

print("Proyecto:", PROJECT_ROOT)
print("Reports:", REPORTS_DIR)
print("Models:", MODELS_DIR)


Proyecto: /Users/alexandralozano/dp261-g1
Reports: /Users/alexandralozano/dp261-g1/reports
Models: /Users/alexandralozano/dp261-g1/models


## 1. Cargar train completo y test final

`train_full` se usa para entrenar el modelo final. `test_final` se usa una sola vez para medir el desempeño final.


In [9]:
train_full = pd.read_csv(PROCESSED_DIR / "train_full.csv")
test_final = pd.read_csv(PROCESSED_DIR / "test_final.csv")

X_train_full, y_train_full = split_X_y(train_full)
X_test_final, y_test_final = split_X_y(test_final)

print("Train full:", X_train_full.shape, y_train_full.shape)
print("Test final:", X_test_final.shape, y_test_final.shape)
print("Tasa IsBadBuy train:", round(y_train_full.mean(), 4))
print("Tasa IsBadBuy test:", round(y_test_final.mean(), 4))


Train full: (58386, 40) (58386,)
Test final: (14597, 40) (14597,)
Tasa IsBadBuy train: 0.123
Tasa IsBadBuy test: 0.123


## 2. Definir modelo final operativo

La selección final queda fija: `LogisticRegression` tuneado con threshold `0.70`, elegido por impacto económico esperado.


In [10]:
FINAL_MODEL_NAME = "LogisticRegression"
FINAL_SOURCE = "tuned_loaded"
FINAL_THRESHOLD = 0.70
REFERENCE_THRESHOLD = 0.50

models_results_path = REPORTS_DIR / "models_results.csv"
models_results = pd.read_csv(models_results_path)

selected_rows = models_results[
    (models_results["source"] == FINAL_SOURCE) &
    (models_results["model"] == FINAL_MODEL_NAME)
].copy()

if selected_rows.empty:
    raise ValueError(
        f"No encontré {FINAL_MODEL_NAME} con source={FINAL_SOURCE} en {models_results_path}."
    )

selected_model = selected_rows.iloc[0].to_dict()
selected_model_path = selected_model.get("path", MODELS_DIR / "tuned_LogisticRegression.pkl")

final_model_selection = selected_rows.copy()
final_model_selection["selected_threshold"] = FINAL_THRESHOLD
final_model_selection["reference_threshold"] = REFERENCE_THRESHOLD
final_model_selection["selection_reason"] = (
    "Modelo tuneado elegido por su balance técnico. El threshold 0.70 se usa como "
    "umbral operativo porque maximiza el valor económico esperado bajo los supuestos "
    "de negocio, reduciendo falsos positivos frente al threshold 0.50."
)

display_cols = [
    c for c in [
        "source", "model", "recall_cv_mean", "f2_cv_mean", "precision_cv_mean",
        "selected_threshold", "reference_threshold", "path", "selection_reason"
    ]
    if c in final_model_selection.columns
]

display(final_model_selection[display_cols])

final_model_selection.to_csv(REPORTS_DIR / "final_model_selection_operational.csv", index=False)
print("Guardado:", REPORTS_DIR / "final_model_selection_operational.csv")


,source,model,recall_cv_mean,f2_cv_mean,precision_cv_mean,selected_threshold,reference_threshold,selection_reason
1,tuned_loaded,LogisticRegression,0.610976,0.483195,0.263126,0.7,0.5,Modelo tuneado elegido por su balance técnico....


Guardado: /Users/alexandralozano/dp261-g1/reports/final_model_selection_operational.csv


## 3. Cargar modelo tuneado

Si el path fue generado en otra máquina, se resuelve usando también el nombre del archivo dentro de `models/`.


In [11]:
def resolve_model_path(raw_path):
    raw_path = str(raw_path).strip()
    p = Path(raw_path)

    possible_paths = [
        p,
        PROJECT_ROOT / p,
        MODELS_DIR / p,
        MODELS_DIR / p.name,
        MODELS_DIR / "tuned_LogisticRegression.pkl",
        MODELS_DIR / "tuned_logisticregression.pkl",
    ]

    for candidate in possible_paths:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "No pude encontrar el archivo del modelo. Paths probados: "
        + " | ".join(str(x) for x in possible_paths)
    )


resolved_model_path = resolve_model_path(selected_model_path)
best_model = joblib.load(resolved_model_path)

print("Modelo cargado:", FINAL_MODEL_NAME)
print("Path resuelto:", resolved_model_path)
print("Threshold operativo:", FINAL_THRESHOLD)


Modelo cargado: LogisticRegression
Path resuelto: /Users/alexandralozano/dp261-g1/models/tuned_LogisticRegression.pkl
Threshold operativo: 0.7


## 4. Entrenar con todo `train_full`

El modelo y el threshold operativo ya fueron decididos antes de reportar el resultado final.


In [12]:
best_model.fit(X_train_full, y_train_full)

final_model_path = MODELS_DIR / "final_LogisticRegression_threshold_0_70.pkl"
joblib.dump(best_model, final_model_path)

# Copia estándar para compatibilidad con otros notebooks/scripts.
joblib.dump(best_model, MODELS_DIR / "final_model.pkl")

print("Modelo final guardado:", final_model_path)
print("Copia estándar guardada:", MODELS_DIR / "final_model.pkl")


Modelo final guardado: /Users/alexandralozano/dp261-g1/models/final_LogisticRegression_threshold_0_70.pkl
Copia estándar guardada: /Users/alexandralozano/dp261-g1/models/final_model.pkl


## 5. Evaluar en `test_final`

Se aplica el threshold operativo `0.70`. También se guarda una comparación contra `0.50` como referencia técnica.


In [13]:
def get_positive_scores(model, X):
    if not hasattr(model, "predict_proba"):
        raise AttributeError("El modelo final debe tener predict_proba para aplicar threshold.")

    proba = model.predict_proba(X)

    if proba.ndim == 1:
        return proba

    if proba.shape[1] < 2:
        raise ValueError("predict_proba no devolvió probabilidad para dos clases.")

    return proba[:, 1]


def compute_final_metrics(y_true, y_score, threshold):
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    business_value = (
        tp * BENEFIT_TP +
        fp * COST_FP +
        fn * COST_FN +
        tn * BENEFIT_TN
    )

    return {
        "threshold": float(threshold),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f05": fbeta_score(y_true, y_pred, beta=0.5, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "average_precision": average_precision_score(y_true, y_score),
        "positive_rate": float(np.mean(y_pred)),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "business_value": business_value,
        "avg_value_per_vehicle": business_value / len(y_true),
    }


y_score_test = get_positive_scores(best_model, X_test_final)

threshold_comparison = pd.DataFrame([
    {
        "model": FINAL_MODEL_NAME,
        "source": FINAL_SOURCE,
        "threshold_type": "reference_0_50",
        **compute_final_metrics(y_test_final, y_score_test, REFERENCE_THRESHOLD),
    },
    {
        "model": FINAL_MODEL_NAME,
        "source": FINAL_SOURCE,
        "threshold_type": "operational_0_70",
        **compute_final_metrics(y_test_final, y_score_test, FINAL_THRESHOLD),
    },
])

final_validation_results = threshold_comparison[
    threshold_comparison["threshold_type"] == "operational_0_70"
].copy()

final_validation_results["original_model_path"] = str(resolved_model_path)
final_validation_results["final_model_path"] = str(final_model_path)

display(threshold_comparison)
display(final_validation_results)

threshold_comparison.to_csv(REPORTS_DIR / "final_validation_threshold_comparison.csv", index=False)
final_validation_results.to_csv(REPORTS_DIR / "final_validation_metrics.csv", index=False)

print("Guardado:", REPORTS_DIR / "final_validation_threshold_comparison.csv")
print("Guardado:", REPORTS_DIR / "final_validation_metrics.csv")


,model,source,threshold_type,threshold,recall,f2,precision,f1,f05,roc_auc,average_precision,positive_rate,tp,fp,fn,tn,business_value,avg_value_per_vehicle
0,LogisticRegression,tuned_loaded,reference_0_50,0.5,0.615599,0.487859,0.266586,0.372054,0.30068,0.767779,0.443197,0.283962,1105,3040,690,9762,-4767000,-326.573954
1,LogisticRegression,tuned_loaded,operational_0_70,0.7,0.367131,0.389020,0.510853,0.427229,0.47376,0.767779,0.443197,0.088374,659,631,1136,12171,-1367400,-93.676783


,model,source,threshold_type,threshold,recall,f2,precision,f1,f05,roc_auc,average_precision,positive_rate,tp,fp,fn,tn,business_value,avg_value_per_vehicle,original_model_path,final_model_path
1,LogisticRegression,tuned_loaded,operational_0_70,0.7,0.367131,0.38902,0.510853,0.427229,0.47376,0.767779,0.443197,0.088374,659,631,1136,12171,-1367400,-93.676783,/Users/alexandralozano/dp261-g1/models/tuned_L...,/Users/alexandralozano/dp261-g1/models/final_L...


Guardado: /Users/alexandralozano/dp261-g1/reports/final_validation_threshold_comparison.csv
Guardado: /Users/alexandralozano/dp261-g1/reports/final_validation_metrics.csv


## 6. Matriz de confusión e impacto económico final

Esta tabla traduce el resultado operativo a términos de negocio usando los supuestos definidos en `config.py`.


In [14]:
final_metrics = final_validation_results.iloc[0].to_dict()

confusion_summary = pd.DataFrame([{
    "tn_buenos_correctos": int(final_metrics["tn"]),
    "fp_buenos_marcados_malos": int(final_metrics["fp"]),
    "fn_malos_no_detectados": int(final_metrics["fn"]),
    "tp_malos_detectados": int(final_metrics["tp"]),
}])

impact_summary = pd.DataFrame([
    {
        "case": "TP - Bad Buy detectado",
        "count": int(final_metrics["tp"]),
        "unit_value": BENEFIT_TP,
        "total_value": int(final_metrics["tp"]) * BENEFIT_TP,
    },
    {
        "case": "FP - Auto bueno rechazado",
        "count": int(final_metrics["fp"]),
        "unit_value": COST_FP,
        "total_value": int(final_metrics["fp"]) * COST_FP,
    },
    {
        "case": "FN - Bad Buy no detectado",
        "count": int(final_metrics["fn"]),
        "unit_value": COST_FN,
        "total_value": int(final_metrics["fn"]) * COST_FN,
    },
    {
        "case": "TN - Auto bueno aceptado",
        "count": int(final_metrics["tn"]),
        "unit_value": BENEFIT_TN,
        "total_value": int(final_metrics["tn"]) * BENEFIT_TN,
    },
])

display(confusion_summary)
display(impact_summary)

confusion_summary.to_csv(REPORTS_DIR / "final_confusion_matrix_summary.csv", index=False)
impact_summary.to_csv(REPORTS_DIR / "final_confusion_matrix_business_impact.csv", index=False)

print("Guardado:", REPORTS_DIR / "final_confusion_matrix_summary.csv")
print("Guardado:", REPORTS_DIR / "final_confusion_matrix_business_impact.csv")


,tn_buenos_correctos,fp_buenos_marcados_malos,fn_malos_no_detectados,tp_malos_detectados
0,12171,631,1136,659


,case,count,unit_value,total_value
0,TP - Bad Buy detectado,659,1200,790800
1,FP - Auto bueno rechazado,631,-1800,-1135800
2,FN - Bad Buy no detectado,1136,-900,-1022400
3,TN - Auto bueno aceptado,12171,0,0


Guardado: /Users/alexandralozano/dp261-g1/reports/final_confusion_matrix_summary.csv
Guardado: /Users/alexandralozano/dp261-g1/reports/final_confusion_matrix_business_impact.csv


## 7. Comentario para el reporte

> Se seleccionó `LogisticRegression` tuneado como modelo final por su balance técnico entre recall, F2 y precision. Aunque el threshold estándar `0.50` captura más `Bad Buys`, el análisis económico mostró que genera demasiados falsos positivos, es decir, rechaza muchos autos buenos. Bajo los supuestos de negocio definidos, el threshold `0.70` maximiza el valor económico esperado porque reduce significativamente el costo por oportunidades comerciales perdidas, manteniendo una detección relevante de compras riesgosas. Por ello, la recomendación final es usar `LogisticRegression` tuneado con threshold operativo `0.70`.
